# Ingestion-Query Pipeline + Video Retrieval Test

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, time, os, json, zipfile, random

In [ ]:
def linear_attn_step(q, k, v, decay, eps, S, z):
    qf, kf, vf = F.elu(q)+1, F.elu(k)+1, F.elu(v)+1
    S = decay * S + kf.transpose(-2,-1) @ vf
    z = decay * z + kf.sum(dim=-2)
    out = (qf @ S) / (qf @ z.unsqueeze(-1)).clamp(min=eps)
    return out, S, z

class IngestionLayer(nn.Module):
    def __init__(self, dim=256, hd=64, nh=4):
        super().__init__(); self.hd=hd; self.nh=nh
        self.csa_k=nn.Linear(dim,hd,0); self.csa_v=nn.Linear(dim,hd,0)
        self.csa_q=nn.Linear(dim,hd,0); self.csa_o=nn.Linear(hd,dim,0)
        self.hca_k=nn.Linear(dim,hd,0); self.hca_v=nn.Linear(dim,hd,0)
        self.hca_q=nn.Linear(dim,hd,0); self.hca_o=nn.Linear(hd,dim,0)
        self.swa_q=nn.Linear(dim,dim,0); self.swa_k=nn.Linear(dim,dim,0)
        self.swa_v=nn.Linear(dim,dim,0); self.swa_o=nn.Linear(dim,dim,0)
        self.norm1=nn.LayerNorm(dim); self.norm2=nn.LayerNorm(dim)
        self.ffn=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
        self.gate=nn.Parameter(torch.ones(3))
    def forward(self, h, sc=None, zc=None, sh=None, zh=None):
        B,T,_=h.shape
        qc=self.csa_q(h).view(B,T,1,self.hd).transpose(1,2)
        kc=self.csa_k(h).view(B,T,1,self.hd).transpose(1,2)
        vc=self.csa_v(h).view(B,T,1,self.hd).transpose(1,2)
        if sc is None: sc=torch.zeros(B,1,self.hd,self.hd); zc=torch.zeros(B,1,self.hd)
        oc,sc,zc=linear_attn_step(qc,kc,vc,0.99,1e-6,sc,zc)
        oc=self.csa_o(oc).squeeze(1)
        qh=self.hca_q(h).view(B,T,1,self.hd).transpose(1,2)
        kh=self.hca_k(h).view(B,T,1,self.hd).transpose(1,2)
        vh=self.hca_v(h).view(B,T,1,self.hd).transpose(1,2)
        if sh is None: sh=torch.zeros(B,1,self.hd,self.hd); zh=torch.zeros(B,1,self.hd)
        oh,sh,zh=linear_attn_step(qh,kh,vh,0.999,1e-6,sh,zh)
        oh=self.hca_o(oh).squeeze(1)
        win=min(T,128)
        qs=self.swa_q(h[:,-win:]).reshape(B,win,self.nh,self.hd).transpose(1,2)
        ks=self.swa_k(h[:,-win:]).reshape(B,win,self.nh,self.hd).transpose(1,2)
        vs=self.swa_v(h[:,-win:]).reshape(B,win,self.nh,self.hd).transpose(1,2)
        os_=F.scaled_dot_product_attention(qs,ks,vs,is_causal=True)
        os_=self.swa_o(os_.transpose(1,2).reshape(B,T,-1))
        g=F.softmax(self.gate,0)
        h=self.norm1(h+g[0]*oc+g[1]*oh+g[2]*os_)
        return h+self.ffn(self.norm2(h)),sc,zc,sh,zh

class Retriever(nn.Module):
    def __init__(self,dim=256,nh=4):
        super().__init__()
        self.ca=nn.MultiheadAttention(dim,nh,batch_first=True)
        self.norm=nn.LayerNorm(dim); self.proj=nn.Linear(dim,1,0)
    def forward(self,q,fe):
        a=self.ca(q,fe,fe)[0]; return torch.softmax(self.proj(self.norm(q+a)).squeeze(-1),-1)

class VideoRetrievalModel(nn.Module):
    def __init__(self,n_layers=2,dim=256,hd=64,nh=4,v=8192):
        super().__init__()
        self.embed=nn.Embedding(v,dim)
        self.layers=nn.ModuleList([IngestionLayer(dim,hd,nh) for _ in range(n_layers)])
        self.retriever=Retriever(dim,nh)
        self.fp=nn.Linear(dim,dim,0); self.tp=nn.Linear(dim,dim,0)
    def ingest(self,tok,states=None):
        if states is None: states=[(None,None,None,None) for _ in self.layers]
        h=self.embed(tok); new=[]
        for i,l in enumerate(self.layers):
            sc,zc,sh,zh=states[i]
            h,sc,zc,sh,zh=l(h,sc,zc,sh,zh)
            new.append((sc,zc,sh,zh))
        return self.fp(h.mean(1)),new
    def retrieve(self,q,fe):
        q=self.tp(self.embed(q).mean(1,keepdim=True))
        return self.retriever(q,fe)

In [ ]:
zip_path=os.path.expanduser('~/Desktop/918822019.github.io/data/coco/PAI/COCO2017/annotations_trainval2017.zip')
with zipfile.ZipFile(zip_path,'r') as z:
    with z.open('annotations/captions_train2017.json') as f: data=json.load(f)
ic={}
for a in data['annotations']:
    iid=a['image_id']
    if iid not in ic: ic[iid]=[]
    ic[iid].append(a['caption'].lower())
vs=[(c,i) for i,c in ic.items() if len(c)>=3]
random.shuffle(vs); print(f"Pseudo-videos: {len(vs)}")

In [ ]:
from tokenizers import Tokenizer
tok=Tokenizer.from_file('bpe_tokenizer.json')
BOS,EOS=tok.token_to_id('<bos>'),tok.token_to_id('<eos>')
v=tok.get_vocab_size()
def enc(s): return [BOS]+tok.encode(s).ids[:64]+[EOS]

In [ ]:
m=VideoRetrievalModel(n_layers=2,dim=256,hd=64,nh=4,v=v)
print(f"Params: {sum(p.numel() for p in m.parameters()):,}"); m.eval()
ae=[]; am=[]; st=None; t0=time.time()
for caps,iid in vs[:50]:
    for cap in caps[:5]:
        ti=torch.tensor([enc(cap)],dtype=torch.long)
        emb,st=m.ingest(ti,st)
        ae.append(emb); am.append(cap[:80])
embs=torch.cat(ae,0).unsqueeze(0)
kv=sum(s.element_size()*s.numel() for so in st for s in so if s is not None)/1024
print(f"Ingested {len(ae)} frames, KV cache: {kv:.1f} KB, time: {time.time()-t0:.1f}s")

In [ ]:
def search(q,k=5):
    qt=torch.tensor([enc(q)],dtype=torch.long)
    s=m.retrieve(qt,embs)
    ti=torch.topk(s[0],min(k,s.shape[-1])).indices.tolist()
    return [(am[i],s[0,i].item()) for i in ti]
for q in ['a cat','a person','food','a car','an animal','outside']:
    r=search(q,3)
    print(f'\nQuery: "{q}"')
    for meta,score in r:
        print(f"  [{score:.3f}] {meta}")

## Done
- Ingestion: O(1) KV cache ✅
- Query: cross-attention retrieval ✅
- End-to-end video query pipeline ✅